# 01 — Data Cleaning
## Project: Olist Sales, Customer & BI Platform

**Purpose:** Systematically detect and fix data quality issues before any analysis.  
**Approach:** Document every problem found. Fix only what genuinely affects business questions.  
**Output:** Cleaned CSV files saved to `data/cleaned/` — raw files never touched.

---

### Issues to Investigate
| # | Issue | Table | Expected Severity |
|---|---|---|---|
| 1 | `review_id` duplicates (814) | reviews | 🔴 High |
| 2 | Reviews file encoding (MacRoman) | reviews | 🟡 Medium |
| 3 | Translation file BOM (UTF-8-SIG) | translation | 🟢 Low |
| 4 | NULL delivery dates | orders | 🟡 Medium — expected |
| 5 | NULL product physical attributes | products | 🟢 Low |
| 6 | `not_defined` payment type (3 rows) | payments | 🟢 Low |
| 7 | Geolocation row count discrepancy | geolocation | 🔴 Needs investigation |

## Section 0 — Setup

In [ ]:
import pandas as pd
import numpy as np
import os

# Paths
RAW_DIR     = '../data/raw/'
CLEANED_DIR = '../data/cleaned/'

os.makedirs(CLEANED_DIR, exist_ok=True)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('Setup complete.')
print(f'pandas version : {pd.__version__}')
print(f'numpy version  : {np.__version__}')
print(f'Raw dir        : {os.path.abspath(RAW_DIR)}')
print(f'Cleaned dir    : {os.path.abspath(CLEANED_DIR)}')

## Section 1 — Load All Datasets

**Why we load carefully:**  
- `reviews` must be read with `encoding='latin-1'` to handle Portuguese special characters (ã, ç, ó)
- `translation` has a BOM (Byte Order Mark) — use `encoding='utf-8-sig'` to strip it automatically
- All other files are plain ASCII — `utf-8` works fine

In [ ]:
# Load all 9 datasets with appropriate encodings
orders      = pd.read_csv(RAW_DIR + 'olist_orders_dataset.csv')
items       = pd.read_csv(RAW_DIR + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(RAW_DIR + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(RAW_DIR + 'olist_order_reviews_dataset.csv',
                          encoding='latin-1')          # Handles Portuguese chars
customers   = pd.read_csv(RAW_DIR + 'olist_customers_dataset.csv')
sellers     = pd.read_csv(RAW_DIR + 'olist_sellers_dataset.csv')
products    = pd.read_csv(RAW_DIR + 'olist_products_dataset.csv')
geolocation = pd.read_csv(RAW_DIR + 'olist_geolocation_dataset.csv')
translation = pd.read_csv(RAW_DIR + 'product_category_name_translation.csv',
                          encoding='utf-8-sig')         # Strips BOM automatically

# Store as a dict for easy iteration
all_dfs = {
    'orders':      orders,
    'items':       items,
    'payments':    payments,
    'reviews':     reviews,
    'customers':   customers,
    'sellers':     sellers,
    'products':    products,
    'geolocation': geolocation,
    'translation': translation,
}

print('All datasets loaded:')
for name, df in all_dfs.items():
    print(f'  {name:<15}: {len(df):>10,} rows  |  {df.shape[1]} cols')

## Section 2 — NULL Analysis (All Tables)

**Why this matters:** NULLs are not always errors. Some NULLs are expected by design (e.g., delivery date is NULL for canceled orders). We need to understand EACH NULL before deciding what to do with it.

In [ ]:
print('='*65)
print('NULL ANALYSIS — ALL TABLES')
print('='*65)

for name, df in all_dfs.items():
    null_counts = df.isnull().sum()
    null_pct    = (df.isnull().sum() / len(df) * 100).round(2)
    
    # Only show columns that have at least one NULL
    mask = null_counts > 0
    if mask.sum() == 0:
        print(f'\n{name.upper()}: ✅ No NULLs found')
    else:
        print(f'\n{name.upper()} — {mask.sum()} columns with NULLs:')
        for col in null_counts[mask].index:
            print(f'  {col:<45} {null_counts[col]:>7,} NULLs ({null_pct[col]:.1f}%)')

## Section 3 — Duplicate Analysis

In [ ]:
print('='*65)
print('DUPLICATE ANALYSIS')
print('='*65)

# ── 3a. Full-row duplicates across all tables ──
print('\n3a. Full-row duplicates:')
for name, df in all_dfs.items():
    dupes = df.duplicated().sum()
    flag  = '⚠' if dupes > 0 else '✅'
    print(f'  {name:<15}: {dupes:>6,} duplicate rows  {flag}')

# ── 3b. Primary key uniqueness ──
print('\n3b. Primary key uniqueness:')
pk_checks = [
    ('orders',   'order_id'),
    ('customers','customer_id'),
    ('sellers',  'seller_id'),
    ('products', 'product_id'),
    ('reviews',  'review_id'),
    ('translation', 'product_category_name'),
]
for name, col in pk_checks:
    df   = all_dfs[name]
    total    = len(df)
    distinct = df[col].nunique()
    dupes    = total - distinct
    flag     = '✅' if dupes == 0 else f'⚠ {dupes:,} duplicates'
    print(f'  {name}.{col:<35}: {total:,} rows | {distinct:,} distinct  {flag}')

# ── 3c. Composite key check ──
print('\n3c. Composite primary keys:')

items_dupes = items.duplicated(subset=['order_id','order_item_id']).sum()
print(f'  items.(order_id, order_item_id)         : {items_dupes} duplicates  {"✅" if items_dupes==0 else "⚠"}')

pay_dupes = payments.duplicated(subset=['order_id','payment_sequential']).sum()
print(f'  payments.(order_id, payment_sequential) : {pay_dupes} duplicates  {"✅" if pay_dupes==0 else "⚠"}')

## Section 4 — Deep Dive: Review Duplicates

We know `review_id` is not unique (814 duplicates). Before fixing, we need to understand WHY.

In [ ]:
# ── 4a. How many duplicate review_ids? ──
rev_id_dupes = reviews[reviews.duplicated(subset='review_id', keep=False)]
print(f'Rows involved in review_id duplicates: {len(rev_id_dupes):,}')
print(f'Distinct duplicated review_ids       : {rev_id_dupes["review_id"].nunique():,}')

# ── 4b. Are the duplicate review_ids attached to the same order_id? ──
print('\nSample of duplicated review_ids:')
sample = rev_id_dupes.sort_values('review_id').head(10)
print(sample[['review_id','order_id','review_score',
              'review_creation_date','review_answer_timestamp']].to_string(index=False))

# ── 4c. Are the duplicates exact copies, or different rows sharing the same review_id? ──
full_row_dupes_in_reviews = reviews.duplicated().sum()
print(f'\nFull-row duplicates in reviews: {full_row_dupes_in_reviews:,}')
print('Conclusion: same review_id but different order_id = review_id is NOT a reliable PK')
print('Safe join key for reviews: order_id (not review_id)')

## Section 5 — Deep Dive: Geolocation

Phase 2 (CSV row count) showed 1,000,163 rows.  
Phase 3 (DuckDB) showed 19,015 rows with 1.0 uniqueness ratio.  
We need to resolve this discrepancy.

In [ ]:
print(f'Geolocation shape (pandas): {geolocation.shape}')
print(f'Distinct zip prefixes     : {geolocation["geolocation_zip_code_prefix"].nunique():,}')

# Count rows per zip prefix
zip_counts = (geolocation
              .groupby('geolocation_zip_code_prefix')
              .size()
              .reset_index(name='row_count'))

print(f'\nRows-per-zip distribution:')
print(f'  Min rows per zip : {zip_counts["row_count"].min()}')
print(f'  Max rows per zip : {zip_counts["row_count"].max():,}')
print(f'  Avg rows per zip : {zip_counts["row_count"].mean():.1f}')
print(f'  Zips with 1 row  : {(zip_counts["row_count"]==1).sum():,}')
print(f'  Zips with >10 rows : {(zip_counts["row_count"]>10).sum():,}')

# The zip with the most rows
top_zip = zip_counts.loc[zip_counts['row_count'].idxmax()]
print(f'\nZip with most rows: {top_zip["geolocation_zip_code_prefix"]} ({top_zip["row_count"]:,} rows)')

print('\n→ Conclusion: pandas sees 1M rows (correct). DuckDB read zip as integer and deduplicated.')
print('→ Safe fix: pre-aggregate geolocation to one row per zip before any join.')

## Section 6 — Deep Dive: NULL Delivery Dates

We expect NULLs in `order_delivered_customer_date` for non-delivered orders. Verify this assumption.

In [ ]:
# Orders with NULL delivery date — which statuses do they have?
null_delivery = orders[orders['order_delivered_customer_date'].isnull()]

print(f'Orders with NULL delivered date: {len(null_delivery):,}')
print('\nStatus breakdown of those NULL-delivery orders:')
print(null_delivery['order_status'].value_counts().to_string())

# Are there any DELIVERED orders with NULL delivery date? (That would be a real error)
delivered_null = orders[
    (orders['order_status'] == 'delivered') &
    (orders['order_delivered_customer_date'].isnull())
]
print(f'\nDelivered orders with NULL delivery date: {len(delivered_null):,}')
if len(delivered_null) == 0:
    print('✅ All delivered orders have a delivery date.')
    print('→ NULLs are expected and valid — they belong to non-delivered orders only.')
    print('→ Decision: Do NOT fill NULLs. Filter to delivered orders in revenue queries.')
else:
    print(f'⚠ {len(delivered_null)} delivered orders are missing a delivery date — needs investigation.')

## Section 7 — Deep Dive: NULL Product Attributes

In [ ]:
# Physical attribute NULLs in products
physical_cols = ['product_weight_g','product_length_cm',
                 'product_height_cm','product_width_cm']

print('Products with NULL physical attributes:')
for col in physical_cols:
    n = products[col].isnull().sum()
    pct = n / len(products) * 100
    print(f'  {col:<35}: {n:,} ({pct:.1f}%)')

# Are these products actually ordered? (If yes, freight calculation is affected)
products_with_null_weight = products[products['product_weight_g'].isnull()]['product_id']
ordered_null_weight = items[items['product_id'].isin(products_with_null_weight)]

print(f'\nOf those NULL-weight products:')
print(f'  Items ordered from NULL-weight products: {len(ordered_null_weight):,}')

if len(ordered_null_weight) > 0:
    print(f'  Sum of freight_value for those items   : R$ {ordered_null_weight["freight_value"].sum():,.2f}')
    print('  → Note: freight was still charged (from payments), even if product weight is NULL')
    print('  → Decision: keep NULLs — we do NOT use physical dimensions in revenue analysis')

# Note the column name typo
print(f"\n⚠ Column name typo: 'product_name_lenght' (original dataset typo)")
print("  Decision: rename to 'product_name_length' in cleaned output — document the change.")

## Section 8 — Deep Dive: `not_defined` Payment Type

In [ ]:
# Inspect the not_defined payments
not_defined = payments[payments['payment_type'] == 'not_defined']
print(f'Rows with payment_type = not_defined: {len(not_defined)}')
print(not_defined.to_string(index=False))

# Do these orders have other payment records?
not_def_orders = not_defined['order_id'].tolist()
related_payments = payments[payments['order_id'].isin(not_def_orders)]
print(f'\nAll payment records for these {len(not_def_orders)} orders:')
print(related_payments.to_string(index=False))

## Section 9 — Data Type Fixes

**Why this matters:** Pandas reads everything as strings by default. Timestamp columns need to be `datetime` objects so we can calculate delivery days, monthly trends, and YoY comparisons.

In [ ]:
print('Orders dtypes BEFORE fix:')
print(orders.dtypes)
print()

# Timestamp columns in orders
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Timestamp columns in reviews
for col in ['review_creation_date', 'review_answer_timestamp']:
    reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

# Timestamp column in items
items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'], errors='coerce')

print('Orders dtypes AFTER fix:')
print(orders.dtypes)
print('\n✅ Timestamp columns converted to datetime.')

## Section 10 — Apply All Fixes

Summary of every decision made:

In [ ]:
print('='*65)
print('APPLYING CLEANING FIXES')
print('='*65)

# ── Fix 1: Rename typo column in products ──
products_clean = products.rename(columns={'product_name_lenght': 'product_name_length'})
print('\nFix 1 ✅ Renamed product_name_lenght → product_name_length')

# ── Fix 2: Remove duplicate review rows (full-row duplicates only) ──
# We do NOT deduplicate on review_id — same review_id can belong to different orders.
# We remove only fully identical rows.
reviews_before = len(reviews)
reviews_clean  = reviews.drop_duplicates()
reviews_after  = len(reviews_clean)
print(f'\nFix 2 ✅ Removed {reviews_before - reviews_after:,} fully-duplicate review rows')
print(f'         reviews: {reviews_before:,} → {reviews_after:,} rows')

# ── Fix 3: Remove not_defined payments (0 value, 3 rows) ──
payments_before = len(payments)
payments_clean  = payments[payments['payment_type'] != 'not_defined'].copy()
payments_after  = len(payments_clean)
print(f'\nFix 3 ✅ Removed {payments_before - payments_after:,} not_defined payment rows (all BRL 0.00)')
print(f'         payments: {payments_before:,} → {payments_after:,} rows')

# ── Fix 4: Pre-aggregate geolocation to one row per zip ──
geo_before = len(geolocation)
geolocation_clean = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat   = ('geolocation_lat',   'mean'),
        geolocation_lng   = ('geolocation_lng',   'mean'),
        geolocation_city  = ('geolocation_city',  'first'),
        geolocation_state = ('geolocation_state', 'first'),
    )
)
geo_after = len(geolocation_clean)
print(f'\nFix 4 ✅ Aggregated geolocation: {geo_before:,} → {geo_after:,} rows (one per zip prefix)')

# ── Fix 5: Add derived columns to orders for convenience ──
# These help analysis — we compute once and save.
orders_clean = orders.copy()
orders_clean['purchase_year']    = orders_clean['order_purchase_timestamp'].dt.year
orders_clean['purchase_month']   = orders_clean['order_purchase_timestamp'].dt.month
orders_clean['purchase_quarter'] = orders_clean['order_purchase_timestamp'].dt.quarter
orders_clean['purchase_ym']      = orders_clean['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Delivery days (only for delivered orders)
orders_clean['delivery_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_purchase_timestamp']
).dt.days

# On-time flag (1 = on time, 0 = late, NULL = not delivered)
orders_clean['delivered_on_time'] = np.where(
    orders_clean['order_status'] != 'delivered', np.nan,
    np.where(
        orders_clean['order_delivered_customer_date'] <=
        orders_clean['order_estimated_delivery_date'], 1, 0
    )
)

print(f'\nFix 5 ✅ Added derived columns to orders:')
print('         purchase_year, purchase_month, purchase_quarter, purchase_ym')
print('         delivery_days, delivered_on_time')

# Keep other tables clean (no issues found needing fixes)
items_clean       = items.copy()    # timestamps already converted above
customers_clean   = customers.copy()
sellers_clean     = sellers.copy()
translation_clean = translation.copy()

print('\n✅ All fixes applied.')

## Section 11 — Validation: Confirm Fixes Worked

In [ ]:
print('='*65)
print('VALIDATION REPORT')
print('='*65)

# Validate Fix 1 — column rename
assert 'product_name_length'  in products_clean.columns, 'Fix 1 failed'
assert 'product_name_lenght' not in products_clean.columns, 'Fix 1 failed'
print('\nFix 1 — Typo rename        : ✅ Validated')

# Validate Fix 2 — no full-row duplicates in reviews
assert reviews_clean.duplicated().sum() == 0, 'Fix 2 failed'
print('Fix 2 — Review duplicates  : ✅ Validated (0 full-row dupes)')

# Validate Fix 3 — no not_defined payments
assert (payments_clean['payment_type'] == 'not_defined').sum() == 0, 'Fix 3 failed'
print('Fix 3 — not_defined removed: ✅ Validated')

# Validate Fix 4 — geolocation one-row-per-zip
assert geolocation_clean.duplicated(subset='geolocation_zip_code_prefix').sum() == 0, 'Fix 4 failed'
print(f'Fix 4 — Geolocation deduped: ✅ Validated ({len(geolocation_clean):,} unique zip rows)')

# Validate Fix 5 — new columns exist
for col in ['purchase_year','purchase_month','purchase_quarter',
            'purchase_ym','delivery_days','delivered_on_time']:
    assert col in orders_clean.columns, f'Fix 5 failed: {col} missing'
print('Fix 5 — Derived columns    : ✅ Validated')

# Quick sanity on delivery days
delivered_orders = orders_clean[orders_clean['order_status']=='delivered']
print(f'\nDelivery days (delivered orders):')
print(f'  Min : {delivered_orders["delivery_days"].min():.0f} days')
print(f'  Max : {delivered_orders["delivery_days"].max():.0f} days')
print(f'  Mean: {delivered_orders["delivery_days"].mean():.1f} days')

# On-time rate
on_time_rate = delivered_orders['delivered_on_time'].mean() * 100
print(f'\nOn-time delivery rate: {on_time_rate:.1f}%')

print('\n✅ All validations passed.')

## Section 12 — Export Cleaned Data

In [ ]:
print('Saving cleaned datasets to data/cleaned/ ...')

cleaned_dfs = {
    'orders_clean.csv':      orders_clean,
    'items_clean.csv':       items_clean,
    'payments_clean.csv':    payments_clean,
    'reviews_clean.csv':     reviews_clean,
    'customers_clean.csv':   customers_clean,
    'sellers_clean.csv':     sellers_clean,
    'products_clean.csv':    products_clean,
    'geolocation_clean.csv': geolocation_clean,
    'translation_clean.csv': translation_clean,
}

for filename, df in cleaned_dfs.items():
    path = CLEANED_DIR + filename
    df.to_csv(path, index=False, encoding='utf-8')
    size_kb = os.path.getsize(path) / 1024
    print(f'  ✅ {filename:<35} {len(df):>10,} rows  |  {size_kb:>8,.1f} KB')

print('\nAll cleaned files saved. Raw files untouched.')

## Section 13 — Cleaning Summary

A permanent record of every decision made and why.

In [ ]:
summary = """
╔══════════════════════════════════════════════════════════════╗
║              DATA CLEANING DECISION LOG                      ║
║              Olist Sales & BI Platform — Phase 4             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Fix 1  Column rename (products)                             ║
║         product_name_lenght → product_name_length            ║
║         Reason: Original typo. Renamed for clarity.          ║
║                                                              ║
║  Fix 2  Remove full-row duplicate reviews                    ║
║         review_id NOT deduplicated — same ID can map to      ║
║         different orders. Full-row dupes removed only.       ║
║         Safe join key for reviews: order_id                  ║
║                                                              ║
║  Fix 3  Remove not_defined payment rows (3 rows, BRL 0)      ║
║         Reason: Zero-value undefined payments — no revenue   ║
║         impact. Excluded to avoid confusion.                 ║
║                                                              ║
║  Fix 4  Aggregate geolocation to one row per zip prefix      ║
║         Reason: Multiple entries per zip cause join fan-out  ║
║         Method: mean(lat/lng), first(city/state) per zip     ║
║                                                              ║
║  Fix 5  Add derived columns to orders                        ║
║         purchase_year, purchase_month, purchase_quarter      ║
║         purchase_ym (YYYY-MM), delivery_days, on_time flag   ║
║         Reason: Avoids recomputing these in every query      ║
║                                                              ║
║  NOT FIXED (intentional decisions):                          ║
║  • NULL delivery dates for non-delivered orders → expected   ║
║  • NULL product physical dimensions → not used in revenue    ║
║  • NULL review comments → text field, optional by design     ║
║  • customer_id vs customer_unique_id → design feature,       ║
║    not a data error. Documented. Use unique_id for counts.   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(summary)